# Correlation and Covariance with Banking Operations

**Dataset:** `banking_operations.csv`  
**Tools:** pandas, NumPy, Matplotlib and topic-specific statistical/ML functions  

This notebook explains the concept in simple terms and connects every calculation to banking operations.

## 1. Core ideas

Covariance shows whether two numerical variables tend to move in the same or opposite directions. Correlation standardizes covariance to a scale from -1 to +1, making strength easier to compare.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

df = pd.read_csv("banking_operations.csv")
df["Transaction_Date"] = pd.to_datetime(df["Transaction_Date"])

print("Dataset shape:", df.shape)
display(df.head())

## 2. Select numerical variables

The dataset contains transaction amount and balance after transaction. We inspect them before calculating relationships.

In [ ]:
numeric_df = df[["Amount", "Balance_After_Transaction"]].copy()
display(numeric_df.describe())

## 3. Covariance with pandas

Positive covariance means the variables tend to move together. Negative covariance means they tend to move in opposite directions. Its size depends on the units.

In [ ]:
covariance_matrix = numeric_df.cov()
display(covariance_matrix)

covariance_value = numeric_df["Amount"].cov(numeric_df["Balance_After_Transaction"])
print(f"Covariance: {covariance_value:,.2f}")

## 4. Correlation with pandas

Pearson correlation measures linear association. A value near zero means little linear association, not necessarily no relationship of any kind.

In [ ]:
correlation_matrix = numeric_df.corr(method="pearson")
display(correlation_matrix)

correlation_value = numeric_df["Amount"].corr(numeric_df["Balance_After_Transaction"])
print(f"Pearson correlation: {correlation_value:.3f}")

## 5. Scatter plot

The plot helps reveal outliers, clusters and non-linear shapes that one coefficient can hide.

In [ ]:
colors_by_status = df["Status"].map({"Completed": "#27AE60", "Pending": "#F2C94C", "Failed": "#EB5757"})
plt.figure(figsize=(8, 5))
plt.scatter(df["Amount"], df["Balance_After_Transaction"], c=colors_by_status, alpha=0.8)
plt.title("Transaction Amount vs Balance After Transaction")
plt.xlabel("Transaction Amount")
plt.ylabel("Balance After Transaction")
plt.grid(alpha=0.2)
plt.show()

## 6. Correlation within operational groups

An overall relationship can hide differences between transaction types or account types.

In [ ]:
correlation_by_type = df.groupby("Transaction_Type").apply(
    lambda group: group["Amount"].corr(group["Balance_After_Transaction"]),
    include_groups=False
).rename("Correlation")
display(correlation_by_type.to_frame())

## 7. Rank correlation

Spearman correlation measures whether values generally move in the same ranked direction and is less dependent on a straight-line relationship.

In [ ]:
comparison = pd.Series({
    "Pearson - linear association": numeric_df.corr(method="pearson").iloc[0, 1],
    "Spearman - rank association": numeric_df.corr(method="spearman").iloc[0, 1]
})
display(comparison.to_frame("Correlation"))

## Banking interpretation and cautions

Correlation can support exploratory analysis, feature selection and risk monitoring. It does not prove causation. Review the scatter plot, sample size, outliers, time effects and group differences before making a business decision.